# Langfuse Basics — Open-Source LLM Observability (Week 6 companion)

**Goal:** get hands-on with Langfuse's core building blocks -- traces, observations, scoring,
datasets/experiments, and prompt management -- using the *same concepts* from today's LangSmith
section, so you can translate between the two in an interview without missing a beat.

This notebook is written so it **runs top to bottom with zero errors even without a Langfuse
account**: every cell that only needs the SDK (decorators, client setup) runs for real; every
cell that needs a live Langfuse project (creating datasets, running experiments) checks
`auth_check()` first and explains what would happen if it ran.

> **Where this fits:** the main [1. WEE6_RAG_Optimization_Live_Session_Master.ipynb](<./1.%20WEE6_RAG_Optimization_Live_Session_Master.ipynb>)
> notebook covers Multi-Query, Compression, Faithfulness/Relevance, LLM-as-Judge, and LangSmith in
> depth. This file is the deep dive on the *"and here's the open-source alternative"* part.

---


## What Is Langfuse?

1. **Open-source core** — the core tracing and eval engine is open-source: read the code,
   self-host it, or use their managed cloud.
2. **Framework-agnostic** — works with LangChain, LlamaIndex, or plain API calls; it doesn't
   assume any particular framework.
3. **Tracing + evals + prompts** — covers the same ground as LangSmith: traces, datasets,
   evaluators, and prompt management.

> **REMEMBER THIS:** Langfuse being open-source doesn't just mean it's free — it means a company
> can run it entirely inside their own infrastructure, which matters a lot for regulated
> industries like insurance or healthcare.

### LangSmith vs. Langfuse — concept-to-concept translation

| LangSmith concept | Langfuse equivalent |
|---|---|
| Project | Project (same idea — a namespace for traces) |
| Trace | Trace |
| Run | Observation (`span`, `generation`, `retriever`, `tool`, `chain`, `agent`, ...) |
| `@traceable` | `@observe` |
| Dataset + `evaluate()` | Dataset + `run_experiment()` |
| Feedback score | Score (`score_current_trace`, `create_score`) |
| Prompt Hub | Prompt Management (`create_prompt`, `get_prompt`) |
| Compare Runs | Compare experiment runs on a dataset |

> **INTERVIEW Q:** "Which observability tool have you used?" → "I've used both — LangSmith when
> the stack is all LangChain, Langfuse when self-hosting or framework-agnostic tracing matters."


## Step 0 — Install & Configure

```
pip install langfuse
```

Langfuse reads three environment variables (get free keys at https://cloud.langfuse.com, or
self-host with Docker):

- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`
- `LANGFUSE_HOST` (defaults to Langfuse Cloud; point this at your own server if self-hosting)

**This notebook works either way.** If these aren't set, `auth_check()` below returns `False`,
and every live-network cell explains what it would do instead of failing.


In [ ]:
# TODO -- Load .env, then create a Langfuse client and check whether it's connected.
# from langfuse import Langfuse, observe, get_client
# client = get_client()
# AUTH_OK = client.auth_check()
# Print whether LANGFUSE_PUBLIC_KEY is present, the host, and the auth_check() result.
# If NOT AUTH_OK, quiet the 'langfuse' logger (logging.getLogger('langfuse').setLevel(logging.CRITICAL))
# so later cells don't repeat the same warning.


## Core Concepts: Trace & Observation

- A **Trace** is one full request, start to finish (same idea as a LangSmith Trace).
- An **Observation** is one step inside that trace. Langfuse types observations so the UI can
  highlight them: `span` (generic step), `generation` (an LLM call), `retriever`, `tool`,
  `chain`, `agent`, `evaluator`, `guardrail`.

The simplest way to create both is the `@observe` decorator — wrap a function and Langfuse
captures its inputs, outputs, timing, and any exceptions automatically.


In [ ]:
# TODO -- Decorate a plain function with @observe(name='add_numbers') and call it.
# Print the result, and client.get_current_trace_id() -- called INSIDE the decorated function
# returns a real id; called from outside it returns None. Try both and see the difference.


Notice we could read `get_current_trace_id()` even without a connected project — Langfuse never
lets tracing break your actual application logic. If it can't reach a server, the call still
returns the real result; it just has nothing to send.


## Tracing an LLM Call (`as_type="generation"`)

For LLM calls specifically, use `as_type="generation"` and enrich the observation with the model
name, parameters, and token usage via `update_current_generation(...)` — this is what powers
Langfuse's cost and latency dashboards (one of its strongest features vs. LangSmith).


In [ ]:
# TODO -- Trace a real ChatOpenAI call as a 'generation' observation.
# Decorate call_llm(question, context) with @observe(name=..., as_type='generation').
# Inside it: build a SystemMessage + HumanMessage prompt, call the llm, then call
# client.update_current_generation(model=..., input=..., output=..., usage_details={...})
# using response.response_metadata['token_usage'].
# Test it on a short insurance question + a one-line context string.


## Automatic Tracing: the LangChain Integration

Everything above used **manual** tracing (`@observe`) -- exactly like LangSmith's `@traceable`.
Langfuse also has an **automatic** side, mirroring LangSmith's "set an env var and every
LangChain call is traced for free": pass a `CallbackHandler` into any LangChain `.invoke(...)`
call and Langfuse creates the trace for you, no decorator needed.

> **REMEMBER THIS:** Same two-mode story as LangSmith -- automatic for LangChain components,
> `@observe` for the custom glue code around them. Most real projects mix both.


In [ ]:
# TODO -- Trace a LangChain call WITHOUT @observe, using Langfuse's automatic integration.
# from langfuse.langchain import CallbackHandler
# handler = CallbackHandler()
# llm.invoke([...], config={'callbacks': [handler]})
# Compare: how is this different from the @observe cell above? What did you NOT have to write?


## Nested Observations: a Tiny Traced RAG Pipeline

Real value shows up once you nest observations — a `retriever` step feeding a `generation` step,
all inside one `chain` trace. Calling an `@observe`-decorated function from inside another
`@observe`-decorated function nests them automatically (no manual parent/child wiring needed).

We use a tiny in-memory knowledge base here (not the full PDF pipeline) so this notebook stays
fast and focused on Langfuse mechanics — the full retrieval pipeline is in the master notebook.


In [ ]:
# TODO -- Build a tiny in-memory KNOWLEDGE_BASE (3-4 short insurance-policy sentences).
# 1. retrieve_context(question, k=1) -- @observe(as_type='retriever') -- rank KNOWLEDGE_BASE by
#    keyword overlap with the question, call client.update_current_span(output=top), return the text.
# 2. traced_rag_pipeline(question) -- @observe(as_type='chain') -- call retrieve_context() then
#    call_llm(), capture trace_id = client.get_current_trace_id() WHILE STILL INSIDE the function,
#    and return {'question','context','answer','trace_id'}.
# Run it and print all four fields.


## Sessions, Users & Tags

Group traces by conversation (`session_id`), by end user (`user_id`), or filter later with
`tags` — use the `propagate_attributes` context manager around any traced call.


In [ ]:
# TODO -- Group a traced call by session/user/tags.
# from langfuse import propagate_attributes
# with propagate_attributes(session_id=..., user_id=..., tags=[...]):
#     result = traced_rag_pipeline(...)
# Print the answer.


## Scoring: Faithfulness & Relevance on a Trace

Same two metrics from Part 2 of today's session, now attached directly to a Langfuse trace with
`score_current_trace(...)` — this is what feeds Langfuse's quality dashboards over time.


In [ ]:
# TODO -- Score a trace with your own LLM-as-judge functions.
# Reuse (or rewrite) faithfulness_score(context, answer) / relevance_score(question, answer)
# from the master notebook.
# Write scored_rag_pipeline(question) -- @observe(as_type='chain') -- that retrieves, answers,
# scores both metrics, then calls client.score_current_trace(name=..., value=..., comment=...,
# data_type='NUMERIC') once per metric. Return a dict with the scores included.


`score_current_trace` (and `client.create_score(trace_id=..., ...)` for scoring a trace after the
fact -- e.g. from a human review queue) both send network requests, so they no-op safely when
`AUTH_OK` is `False`, same as every other Langfuse call.


## You Don't Have to Hand-Roll Judges: `autoevals` (RAGAS-derived)

The `faithfulness_score` / `relevance_score` functions above are hand-rolled LLM-as-judge prompts
-- great for understanding the mechanics, but in practice most teams reach for a library instead
of maintaining their own judge prompts. [`autoevals`](https://github.com/braintrustdata/autoevals)
ships the exact metrics RAGAS made popular -- `Faithfulness` and `AnswerRelevancy` -- as ready-made
scorers, and Langfuse has a first-class adapter for them: `create_evaluator_from_autoevals`.

```
pip install autoevals
```

> **WRITE THIS DOWN:** This is the RAGAS connection from today's wrap-up slide made concrete --
> `autoevals` literally implements RAGAS's faithfulness/relevance algorithms so you don't have to.


In [ ]:
# TODO -- Use autoevals' RAGAS-derived scorers instead of hand-rolled judge prompts.
# from openai import OpenAI
# from autoevals import init as autoevals_init
# from autoevals.ragas import Faithfulness, AnswerRelevancy
# autoevals_init(OpenAI())
# IMPORTANT: pass model='gpt-4o-mini' to both scorers -- the library default is a reasoning model
# that rejects the `temperature` param these scorers send, which raises a 400 error.
# Call .eval(input=question, output=answer, context=context) on each and print both scores.


## Datasets & Experiments

Same pattern as LangSmith's Dataset + `evaluate()`: save a set of test questions (with gold
answers) once, then re-run the whole pipeline against it with one call any time you change
something, and compare runs side by side in the UI.

- `create_dataset(name=...)` + `create_dataset_item(..., expected_output=...)` — build the test
  set, this time WITH a gold reference answer per question (most real eval sets have one).
- `run_experiment(data=..., task=..., evaluators=[...])` — run your pipeline over every item and
  score it automatically. We mix **three** evaluators here: our two hand-rolled judges from
  earlier, plus the `autoevals` `Factuality` scorer via `create_evaluator_from_autoevals` -- so
  you can see a custom evaluator and a library evaluator running side by side in the same run.

Then we run it **twice** with two different retrievers -- a deliberately naive baseline (always
returns the first document, ignoring the question) and our keyword-overlap `retrieve_context` --
so there's a real, comparable gap between the two experiment runs in the Langfuse UI, exactly
like the baseline-vs-optimized comparison you already did in LangSmith.

These all need a real Langfuse project, so this cell checks `AUTH_OK` first.


In [ ]:
# TODO -- Datasets + experiments, with a real baseline-vs-optimized comparison.
# 1. Build eval_set: a list of {'question', 'gold_answer'} dicts.
# 2. naive_retrieve(question): a deliberately BAD baseline that ignores the question (e.g. always
#    returns KNOWLEDGE_BASE[0]).
# 3. Two hand-rolled evaluators (faithfulness_evaluator, relevance_evaluator) matching Langfuse's
#    EvaluatorFunction shape: (*, input, output, expected_output, metadata, **kwargs) -> Evaluation.
# 4. from langfuse.experiment import create_evaluator_from_autoevals ; from autoevals import Factuality
#    factuality_evaluator = create_evaluator_from_autoevals(Factuality(model='gpt-4o-mini'))
# 5. client.create_dataset(name=...) + client.create_dataset_item(..., expected_output=...) per question.
# 6. client.run_experiment(name=..., data=list(dataset.items), task=..., evaluators=[...]) -- run it
#    TWICE, once per retriever, with two different `name=` values so both show up as separate,
#    comparable runs on the same dataset in the Langfuse UI.


## Prompt Management

Langfuse's answer to LangSmith's Prompt Hub: versioned, shared prompts your whole team can pull
from code instead of copy-pasting between notebooks.


In [ ]:
# TODO -- Push and pull a versioned prompt.
# client.create_prompt(name=..., prompt=..., labels=['production'], type='text')
# pulled = client.get_prompt(name, label='production')
# Print pulled.version and pulled.prompt.


## Viewing a Trace & Flushing

Langfuse batches traces and sends them in the background for performance. In a notebook (which
exits quickly), call `client.flush()` to force everything out before you go check the UI.


In [ ]:
# TODO -- Force buffered traces out and print a direct link to a real trace.
# client.flush()
# Use a trace_id you captured EARLIER (from traced_rag_pipeline's return dict -- get_current_trace_id()
# called here, outside any active trace, will just return None).
# client.get_trace_url(trace_id=...)


## Wrap-Up

- **`@observe`** is Langfuse's `@traceable` — decorate a function, get a trace or nested
  observation, and it never breaks your app even when tracing can't reach a server.
- **`CallbackHandler`** is the automatic side, same story as LangSmith's env-var tracing — pass
  it into any LangChain call and get a trace with zero decorators.
- **`as_type`** (`generation`, `retriever`, `tool`, `chain`, ...) is what makes a trace readable
  in the UI — tag your steps honestly and debugging gets much faster.
- **`propagate_attributes`** groups traces by session/user/tags — essential once you have real
  traffic, not just a test set.
- **`autoevals`** is the RAGAS connection made concrete — you don't have to hand-roll every judge
  prompt; `create_evaluator_from_autoevals` plugs a library scorer straight into `run_experiment`
  alongside your own custom evaluators.
- **Scores, datasets, and experiments** map directly onto what you already know from LangSmith:
  same job, same day-to-day workflow, different vendor. Running the SAME dataset through two
  different retrievers and comparing the runs in the UI is exactly how you'd defend "I improved
  it" with a number, in either tool.
- **Prompt management** solves the exact "good prompt buried in one notebook cell" problem from
  today's LangSmith section.

> **REMEMBER THIS:** Rule of thumb from the slides — all-in on LangChain and want the path of
> least resistance → LangSmith. Need self-hosting, data residency, or you're framework-agnostic
> → Langfuse. Now you've actually written code against both.

**To try this live:** sign up free at https://cloud.langfuse.com, drop the three env vars into
your `.env`, and re-run this notebook — every guarded cell above will execute for real and you'll
see traces, scores, datasets, and prompts appear in the dashboard.
